Importing Libraries

In [35]:
import pandas as pd
import numpy as np
from  sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler , OrdinalEncoder, FunctionTransformer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import  r2_score, root_mean_squared_error
# from joblib
import warnings


load data

In [10]:
train_dir = pd.read_csv("C:\\Users\\Sepehr\\Downloads\\playground-series-s5e10\\train.csv")
test_dir = pd.read_csv("C:\\Users\\Sepehr\\Downloads\\playground-series-s5e10\\test.csv")

train_dir.head()

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56


In [11]:
train_dir.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517754 entries, 0 to 517753
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      517754 non-null  int64  
 1   road_type               517754 non-null  object 
 2   num_lanes               517754 non-null  int64  
 3   curvature               517754 non-null  float64
 4   speed_limit             517754 non-null  int64  
 5   lighting                517754 non-null  object 
 6   weather                 517754 non-null  object 
 7   road_signs_present      517754 non-null  bool   
 8   public_road             517754 non-null  bool   
 9   time_of_day             517754 non-null  object 
 10  holiday                 517754 non-null  bool   
 11  school_season           517754 non-null  bool   
 12  num_reported_accidents  517754 non-null  int64  
 13  accident_risk           517754 non-null  float64
dtypes: bool(4), float64(

In [12]:
train_dir.isnull().sum()

id                        0
road_type                 0
num_lanes                 0
curvature                 0
speed_limit               0
lighting                  0
weather                   0
road_signs_present        0
public_road               0
time_of_day               0
holiday                   0
school_season             0
num_reported_accidents    0
accident_risk             0
dtype: int64

In [13]:

test_dir.isnull().sum()

id                        0
road_type                 0
num_lanes                 0
curvature                 0
speed_limit               0
lighting                  0
weather                   0
road_signs_present        0
public_road               0
time_of_day               0
holiday                   0
school_season             0
num_reported_accidents    0
dtype: int64

In [14]:
train_dir.columns

Index(['id', 'road_type', 'num_lanes', 'curvature', 'speed_limit', 'lighting',
       'weather', 'road_signs_present', 'public_road', 'time_of_day',
       'holiday', 'school_season', 'num_reported_accidents', 'accident_risk'],
      dtype='object')

In [15]:
target_col = "accident_risk"
test_ids = test_dir["id"] if "id" in test_dir.columns else np.arange(len(test_dir))



In [16]:
def clean_data(df):
    if "time_of_day" in df.columns:
        order = ["morning", "afternoon", "evening", "night"]
        df["time_of_day"] = pd.Categorical(df["time_of_day"], categories=order, ordered=True)
    return df

train = clean_data(train_dir)
test = clean_data(test_dir)

In [25]:

time_order = {"morning": 0, "afternoon": 1, "evening": 2, "night": 3}
for df in [train, test]:
    if "time_of_day" in df.columns:
        df["time_of_day_num"] = df["time_of_day"].map(time_order)

In [26]:
X = train.drop(columns=[target_col, "id"])
y = train[target_col]

X_test = test_dir.drop(columns=["id"])


In [27]:
numeric_features = ["num_lanes", "curvature", "speed_limit", "num_reported_accidents", "time_of_day_num"]
categorical_features = ["road_type", "lighting", "weather"]
boolean_features = ["road_signs_present", "public_road", "holiday", "school_season"]
ordinal_features = [] 


In [28]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

time_order = ["morning", "afternoon", "evening", "night"]


bool_transformer = Pipeline([
    ("to_int", FunctionTransformer(lambda df: df.astype(int), validate=False))
])

In [30]:
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
     ("bool", bool_transformer, boolean_features)
], remainder="drop")

In [31]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
regressor = RandomForestRegressor(random_state=42)
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", regressor)
])


In [55]:
model.fit(X_train, y_train)

y_pred =  model.predict(X_val)  

In [57]:
rmse = root_mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation R2: {r2:.4f}")

Validation RMSE: 0.0595
Validation R2: 0.8717


In [58]:

model.fit(X, y)
test_preds = model.predict(X_test)


In [59]:

submission = pd.DataFrame({"id": test_ids, "accident_risk": np.round(test_preds, 3)})
submission.to_csv("test_predic tions.csv", index=False)
print("Predictions saved to test_predictions.csv")


Predictions saved to test_predictions.csv


In [60]:
print(submission.shape)

(172585, 2)
